In [ ]:
import joblib
import os
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import lightgbm as lgb
import pyarrow.parquet as pq
from functionality.models import ChemBertaBinaryClassifierLightning, CNNBinaryClassifierLightning, MolFormerXLBinaryClassifierLightning

HSA

In [ ]:
#Load models
protein_name = 'HSA'
path = f"../trained_models/{protein_name}/{protein_name}"
cnn_model = CNNBinaryClassifierLightning.load_from_checkpoint(os.path.join(path, "_CNN.ckpt"))
chembert_model = ChemBertaBinaryClassifierLightning.load_from_checkpoint(os.path.join(path, "_ChemBert.ckpt"))
molformer_model = MolFormerXLBinaryClassifierLightning.load_from_checkpoint(path, "_MolFormer.ckpt")
lightgbm_model = lgb.Booster(model_file=f"../trained_models/{protein_name}/{protein_name}_lightgbm.txt")


In [ ]:
#Load validation sets
def load_val(model_name, protein_name, path = "../intermediates/embeddings/"):
    table = pq.read_table(os.path.join(path, f"{protein_name}_{model_name}_val.parquet"))

    df = table.to_pandas()

    # Extract embeddings (all columns except 'label')
    features = df.drop(columns=["label"]).values 
    labels = df["label"].values 
    return features, labels

chembert_features, labels = load_val('ChemBert', protein_name)
molformer_features, _ = load_val('MolFormer', protein_name)
cnn_features, _ = load_val('MolFormer', 'HSA', protein_name)
lightgbm_features, _ = load_val('lightgbm', protein_name)

In [ ]:
assert np.array_equal(cnn_labels, chembert_labels)
assert np.array_equal(lightgbm_labels, molformer_labels)
assert np.array_equal(chembert_labels, lightgbm_labels)

In [ ]:
cnn_val_preds = cnn_model.predict(cnn_features)
chembert_val_preds = chembert_model.predict(chembert_features)
molformer_val_preds = molformer_model.predict(molformer_features)
lightgbm_val_preds = lightgbm_model.predict(lightgbm_features)

meta_features = np.column_stack((cnn_val_preds, chembert_val_preds, molformer_val_preds, lightgbm_val_preds))

In [ ]:
meta_model = RandomForestClassifier(n_estimators=100, random_state=42)
meta_model.fit(meta_features, labels)

joblib.dump(meta_model, './intermediates/inference/random_forest_meta_model.pkl')

In [ ]:
def weighted_ensemble(predictions, weights):
    
    weighted_sum = np.zeros_like(predictions[0])
    for pred, weight in zip(predictions, weights):
        weighted_sum += pred * weight
    return weighted_sum / np.sum(weights)

In [ ]:
#Test pipeline
cnn_test_features, _ = load_test('CNN', 'HSA', 'encoded_smiles')
chembert_test_features, _ = load_test('ChemBert', 'HSA', 'embeddings')
molformer_test_features, _ = load_test('MolFormer', 'HSA', 'embeddings')
lightgbm_test_features, test_labels = load_test('lightgbm', 'HSA', 'fingerprints')

#Predictions
cnn_test_preds = cnn_model.predict(cnn_test_features)
chembert_test_preds = chembert_model.predict(chembert_test_features)
molformer_test_preds = molformer_model.predict(molformer_test_features)
lightgbm_test_preds = lightgbm_model.predict(lightgbm_test_features)

#Weights apply - for logistic regression
weights = meta_model.coef_[0]  
final_test_preds = weighted_ensemble(
    [cnn_test_preds, chembert_test_preds, molformer_test_preds, lightgbm_test_preds],
    weights
)
#for tree-based model
meta_model = joblib.load('../intermediates/inference/random_forest_meta_model.pkl')
new_meta_features = np.column_stack((
    cnn_model.predict(cnn_test_features),
    chembert_model.predict(chembert_test_features),
    molformer_model.predict(molformer_test_features),
    lightgbm_model.predict(lightgbm_test_features)
))

# Use the meta-model to make final predictions
final_predictions = meta_model.predict(new_meta_features)
